# t-SNE: maps of high-dimensional data
**Machine Learning · Week 8 (The rest of the classical toolbox) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w8d4.html

Each digit image is a point in 64 dimensions. PCA (week 6) finds the best flat *view* of that cloud.
**t-SNE** instead draws a *map*: it places points on a page so that each point's close neighbors in 64
dimensions stay its close neighbors on the page.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

digits = load_digits()
X = digits.data / 16.0
y = digits.target
colors = np.array(["#ff8a50", "#5577f5", "#17a98c", "#e45fa3", "#7b5cfa", "#f0ac2e", "#5aa9f2", "#c08a5a", "#2c2842", "#9bd36b"])

t0 = time.perf_counter()
emb = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X)
print(f"t-SNE of {len(X)} digits took {time.perf_counter() - t0:.1f} s")
pcs = PCA(n_components=2).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), layout="constrained")
for ax, Z, title in [(axes[0], pcs, "PCA: the best flat view"), (axes[1], emb, "t-SNE: a map of neighbors")]:
    ax.scatter(Z[:, 0], Z[:, 1], c=colors[y], s=6)
    for d in range(10):                                   # write each digit at the middle of its points
        cx, cy = np.median(Z[y == d], axis=0)
        ax.text(cx, cy, str(d), fontsize=16, weight="bold", ha="center", va="center",
                bbox=dict(boxstyle="circle", fc="white", ec=colors[d], alpha=0.85))
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## The one knob: perplexity
Perplexity is roughly how many neighbors each point pays attention to. Small: only the very nearest, so the
map breaks into many little clumps. Large: wider neighborhoods, so more of the global layout survives.
(On 600 digits, to keep it quick.)

In [ ]:
sub = np.random.default_rng(0).choice(len(X), 600, replace=False)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
for ax, perp in zip(axes, [3, 30, 150]):
    Z = TSNE(perplexity=perp, random_state=0).fit_transform(X[sub])
    ax.scatter(Z[:, 0], Z[:, 1], c=colors[y[sub]], s=8)
    ax.set_title(f"perplexity {perp}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## How not to read a t-SNE map
Three groups in 2D: a tight one, a wide one, and one far away from both. Their real sizes and distances are
very different. t-SNE keeps the *groups*, but in the map the sizes come out similar and the gaps between
groups no longer mean much.

In [ ]:
rng = np.random.default_rng(1)
A = rng.normal([0, 0], 0.2, (150, 2))       # tight
B = rng.normal([3, 0], 1.0, (150, 2))       # wide
Cc = rng.normal([15, 0], 0.5, (150, 2))     # far away
G = np.vstack([A, B, Cc])
lab = np.repeat([0, 1, 2], 150)
Z = TSNE(perplexity=30, random_state=0).fit_transform(G)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4), layout="constrained")
ax1.scatter(G[:, 0], G[:, 1], c=colors[lab], s=8)
ax1.set_title("The real data")
ax1.set_aspect("equal")
ax2.scatter(Z[:, 0], Z[:, 1], c=colors[lab], s=8)
ax2.set_title("Its t-SNE map: similar sizes, misleading gaps")
plt.show()

## Your turn: same data, different seeds
Run t-SNE on the 600 digits (`X[sub]`) from a random start (`init="random"`; by default t-SNE starts from
the PCA view, which makes its maps very repeatable) with `random_state` 0, 1 and 2, and plot the three maps
side by side. What stays the same, and what changes?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
for ax, seed in zip(axes, [0, 1, 2]):
    Z = None   # TODO: TSNE(perplexity=30, init="random", random_state=seed).fit_transform(X[sub])
    # TODO: scatter Z colored by colors[y[sub]], and title the plot with the seed
plt.show()

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
for ax, seed in zip(axes, [0, 1, 2]):
    Z = TSNE(perplexity=30, init="random", random_state=seed).fit_transform(X[sub])
    ax.scatter(Z[:, 0], Z[:, 1], c=colors[y[sub]], s=8)
    ax.set_title(f"random_state = {seed}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()
print("The islands (which digits clump together) stay; where they sit, and which way up, changes.")